In [ ]:
import pandas as pd

In [ ]:
# Load your data
my_table = pd.read_csv(r'C:\Users\Administrator\Desktop\Clubed.csv')
table_227 = pd.read_csv(r'C:\Users\Administrator\Desktop\Data_227.csv')

# Prepare keys for matching
my_table['match_key'] = (my_table['Managerid'].astype(str) + '_' +
                         my_table['symbol'] + my_table['ExpiryDate'] +
                         my_table['SttrikePrice'] + my_table['OptionType'] + '_' +
                         my_table['ExchangeTradeTime'].astype(str))

table_227['match_key'] = (table_227['CLIENTID'].astype(str) + '_' +
                           table_227['SCRIPT_SYMBOL'] + '_' +
                           table_227['TRADE_DATE1'].astype(str))

# Merge the DataFrames
merged = pd.merge(my_table, table_227, on='match_key', how='outer', suffixes=('_my', '_227'))

# Select relevant columns for output
matched = merged[merged['CLIENTID'].notna()]
unmatched = merged[merged['CLIENTID'].isna()]

# Select the columns you want to save
matched_result = matched[['Managerid', 'symbol', 'expirydate', 'strikeprice', 'optiontype', 
                          'Exchangetradetime', 'BuyQuantity', 'sellQuantity', 
                          'BuyAmount', 'SellAmount']]

unmatched_result = unmatched[['Managerid', 'symbol', 'expirydate', 'strikeprice', 'optiontype', 
                              'Exchangetradetime', 'CLIENTID', 'SCRIPT_SYMBOL', 'TRADE_DATE1']]

# Save results to CSV
matched_result.to_csv('matched_data.csv', index=False)
unmatched_result.to_csv('unmatched_data.csv', index=False)

print("Data processing complete. Matched and unmatched data saved to CSV files.")


In [ ]:
# Load and clean Data227
Data_227 = pd.read_csv(r'C:\Users\Administrator\Desktop\Data_227.csv')
Data_227.columns = Data_227.columns.str.strip().str.replace('[', '').str.replace(']', '')

# Filter out rows with unwanted CompCon values
Data_227 = Data_227[~Data_227['CompCon'].str.startswith('NSE_CASH')]
substrings_to_remove = ['TOC NSE Exchange', 'STT', 'TOC BSE Exchange', 'STAMP DUTY', 'SERVICE TAX']
pattern = '|'.join(substrings_to_remove)
Data_227 = Data_227[~Data_227['CompCon'].str.contains(pattern, case=False, na=False)]

# Filter out rows containing 'EF' or 'IO' and create 'Token'
data227_filtered = Data_227[~Data_227['SCRIP_SYMBOL'].str.contains('EF|IO')].copy()
data227_filtered['Token'] = data227_filtered['CLIENT_ID'] + data227_filtered['SCRIP_SYMBOL']
data227_filtered['Token'] = data227_filtered['Token'].apply(lambda x: x if 'CE' in x or 'PE' in x else 'XX')

# Check the resulting DataFrame
data227_filtered.tail()

In [ ]:
# Print column names to debug
print(data_227.columns)

In [ ]:
import pandas as pd

# Step 1: Read the CSV files
april_data = pd.read_csv(r'C:\Users\Administrator\Desktop\Clubed.csv')  # Replace with your actual file name
data_227 = pd.read_csv(r'C:\Users\Administrator\Desktop\Data_227.csv')      # Replace with your actual file name

# Step 2: Clean column names by stripping whitespace
april_data.columns = april_data.columns.str.strip()
data_227.columns = data_227.columns.str.strip()

# Step 3: Check column names
print("April Data Columns:", april_data.columns)
print("Data 227 Columns:", data_227.columns)

# Step 4: Create SCRIP_SYMBOL in April data by combining relevant columns
april_data['SCRIP_SYMBOL'] = (
    april_data['Symbol'].astype(str) + 
    '-' + 
    april_data['ExpiryDate'].astype(str) + 
    '-' + 
    april_data['ManagerID'].astype(str) + 
    '-' + 
    april_data['OptionType'].astype(str)
)

# Step 5: Rename 'ManagerID' to 'CLIENT_ID' in April data for merging
april_data.rename(columns={'ManagerID': 'CLIENT_ID'}, inplace=True)

# Check column names again after renaming
print("April Data Columns after renaming:", april_data.columns)

# Step 6: Define the columns you want to match on
match_columns = ['CLIENT_ID', 'SCRIP_SYMBOL']  # Matching on CLIENT_ID and combined SCRIP_SYMBOL

# Step 7: Merge the DataFrames based on the matching columns
# Ensure that 'CLIENT_ID' and 'SCRIP_SYMBOL' exist in both DataFrames
merged_data = pd.merge(april_data, data_227, left_on=match_columns, right_on=['CLIENT_ID', 'SCRIP_SYMBOL'], suffixes=('_april', '_227'))

# Step 8: Check for matching quantities
matching_rows = merged_data[(merged_data['BUY_QUANTITY'] == merged_data['BUY_QUANTITY']) &  # Adjust if needed
                             (merged_data['SALE_QUANTITY'] == merged_data['SALE_QUANTITY'])]  # Adjust if needed

# Optionally, display the matching rows
print(matching_rows)

# If you want to save the results to a new CSV file
matching_rows.to_csv('matching_quantities.csv', index=False)